# エラー処理メカニズム
## False/Trueの設定、固定文字列
handle_errors=True

In [1]:
from langchain.chat_models import init_chat_model
import os
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
from rich import print
from dotenv import load_dotenv

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)


class ContactInfo(BaseModel):
    """個人の連絡先情報"""
    name: str = Field(description="氏名")
    email: str = Field(description="メールアドレス")


class EventDetail(BaseModel):
    """イベント詳細"""
    event_name: str = Field(description="イベント名")
    date: str = Field(description="イベント日")


# エージェントを作成 


agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=Union[ContactInfo, EventDetail],
                                 tool_message_content="抽出完了",
                                 handle_errors=True

                                 )
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話番号は12345678912です"
                         "イベント名、忘年会、開催日：2026-07-15")
        ]
    }
)
print(response)
# for msg in response["messages"]:
#     msg.pretty_print()
#     print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話
番号は12345678912ですイベント名、忘年会、開催日：2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='3b483d0a-2727-4ea9-9be2-ab64b05f52a5'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 71,
                    'prompt_tokens': 144,
                    'total_tokens': 215,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.0004275,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0004275,
                        'upstream_inference_prompt_cost': 0.000108,
                        'upstream_inference_completions_cost': 0.0003195
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1785890913-bhth0TxVogBQiBaRxIyf',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fcf64-bb5f-78d3-89ab-0dddea336e4c-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com'},
                    'id': 'call_uIyYAFS57cVgUd5NC4mLngNO',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetail',
                    'args': {'event_name': '忘年会', 'date': '2026-07-15'},
                    'id': 'call_EpXIpEmEy0XbZnxrnQGr05Sf',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 144,
                'output_tokens': 71,
                'total_tokens': 215,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetail) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='ce47aa40-c966-4797-af5d-559bcfd70071',
            tool_call_id='call_uIyYAFS57cVgUd5NC4mLngNO'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetail) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetail',
            id='6f2aacba-2338-4684-b069-059ff88d3fc0',
            tool_call_id='call_EpXIpEmEy0XbZnxrnQGr05Sf'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 29,
                    'prompt_tokens': 287,
                    'total_tokens': 316,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                

handle_errors=False

In [2]:

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=Union[ContactInfo, EventDetail],
                                 tool_message_content="抽出完了",
                                 handle_errors=False

                                 )
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話番号は12345678912です"
                         "イベント名、忘年会、開催日：2026-07-15")
        ]
    }
)
print(response)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetail) when only one is expected.

固定文字列を設定する

In [ ]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=Union[ContactInfo, EventDetail],
                                 tool_message_content="抽出完了",
                                 handle_errors="入力データを確認してください"

                                 )
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話番号は12345678912です"
                         "イベント名、忘年会、開催日：2026-07-15")
        ]
    }
)
print(response)

## ケース2：指定の例外タイプを設定する

In [ ]:


from langchain.agents.structured_output import MultipleStructuredOutputsError, StructuredOutputValidationError

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=Union[ContactInfo, EventDetail],
                                 tool_message_content="抽出完了",
                                 handle_errors=[MultipleStructuredOutputsError, StructuredOutputValidationError]

                                 )
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話番号は12345678912です"
                         "イベント名、忘年会、開催日：2026-07-15")
        ]
    }
)
print(response)

## カスタムエラー処理関数を設定する

In [ ]:
# カスタムエラー処理関数
def custom_error_handler(error: Exception) -> str:
    """カスタムエラーハンドラー"""
    error_str = str(error)
    print(f"捕捉したエラータイプ：{type(error).__name__}")
    print(f"エラー詳細：{error_str}")
    if isinstance(error, StructuredOutputValidationError):
        return "データ形式が正しくありません。フィールドが要件を満たしているか確認してください。"
    elif isinstance(error, MultipleStructuredOutputsError):
        return "複数のレスポンスが検出されました。最も関連性の高いものを選んで返してください。"
    else:
        return f"Error: {error_str}"




agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=Union[ContactInfo, EventDetail],
                                 tool_message_content="抽出完了",
                                 handle_errors=custom_error_handler

                                 )
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話番号は12345678912です"
                         "イベント名、忘年会、開催日：2026-07-15")
        ]
    }
)
print(response)